# Rock Paper Scissors Object Detection

This notebook teaches object detection with YOLOv8 on a 3-class Rock-Paper-Scissors dataset stored in Google Drive.

The lesson is designed to show three things clearly:
- a short baseline training run
- why lowering confidence to `0.01` makes more detections appear
- why training longer for 50 epochs improves the final result

Run the cells in order. Each explanation tells you what the next code cell is doing and why it matters.


## 1. One-time setup

This cell connects Colab to Google Drive, installs Ultralytics, imports the libraries we need, and prepares the dataset path. It also writes the dataset YAML file that YOLO uses to find the images and class names.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

import os
from pathlib import Path
import yaml
from IPython.display import Image, display
from ultralytics import YOLO

base = "/content/drive/MyDrive/rock-paper-scissor.v3-v22.yolov11"
fixed_yaml = "/content/rps_fixed.yaml"

data = {
    "path": base,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": ["paper", "rock", "scissor"],
}

with open(fixed_yaml, "w") as f:
    yaml.dump(data, f)

print(os.listdir(base))
print(Path(fixed_yaml).read_text())


## 2. Train the first model for 10 epochs

This is the baseline experiment. We use the small `yolov8n.pt` model and only 10 epochs so students can compare this weaker starting point with the stronger later result.


In [ ]:
model = YOLO("yolov8n.pt")
results = model.train(
    data=fixed_yaml,
    epochs=10,
    imgsz=640,
    batch=16,
    project="YOLO_Demo",
    name="rps_demo"
)


## 3. Show the confusion matrix for the 10-epoch model

This cell evaluates the first model using the default confidence threshold. It produces the confusion matrix images so students can see the baseline performance before any tuning.


In [ ]:
best_model_10 = YOLO("/content/runs/detect/YOLO_Demo/rps_demo/weights/best.pt")
metrics_10 = best_model_10.val(data=fixed_yaml, plots=True)
val_dir_10 = Path(metrics_10.save_dir)

display(Image(filename=str(val_dir_10 / "confusion_matrix.png")))
display(Image(filename=str(val_dir_10 / "confusion_matrix_normalized.png")))


## 4. Re-run validation with `conf=0.01`

Lowering the confidence threshold keeps more low-confidence predictions instead of discarding them. That usually increases recall and makes the confusion matrix look better, but it can also add a few extra false positives.


In [ ]:
best_model_10 = YOLO("/content/runs/detect/YOLO_Demo/rps_demo/weights/best.pt")
metrics_10_lowconf = best_model_10.val(
    data=fixed_yaml,
    conf=0.01,
    plots=True
)
val_dir_10_low = Path(metrics_10_lowconf.save_dir)

display(Image(filename=str(val_dir_10_low / "confusion_matrix.png")))
display(Image(filename=str(val_dir_10_low / "confusion_matrix_normalized.png")))


## 5. Train a stronger model for 50 epochs

Now we train longer. More epochs give the detector more chances to learn the patterns in the dataset, which usually improves recall and gives a better final model.


In [ ]:
model = YOLO("yolov8n.pt")
results = model.train(
    data=fixed_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    project="YOLO_Demo",
    name="rps_demo_50ep"
)


## 6. Show the confusion matrix for the 50-epoch model

This is the main comparison point for the class. We validate the stronger model using the same low confidence threshold so students can see how much the longer training improved the result.


In [ ]:
best_model_50 = YOLO("/content/runs/detect/YOLO_Demo/rps_demo_50ep/weights/best.pt")
metrics_50 = best_model_50.val(
    data=fixed_yaml,
    conf=0.01,
    plots=True
)
val_dir_50 = Path(metrics_50.save_dir)

display(Image(filename=str(val_dir_50 / "confusion_matrix.png")))
display(Image(filename=str(val_dir_50 / "confusion_matrix_normalized.png")))


## 7. What students should remember

The 10-epoch run is the baseline. Lowering `conf` to `0.01` makes more detections visible, and the 50-epoch run is the version that should be presented as the final stronger model.
